# Week 3 — What's on the desk? Context management

*Course project: a personal assistant for **Alex, 44**: senior backend developer, full-time job, two kids,
retraining as an **AI engineer** while job-hunting.*

> **An LLM only sees what is on its desk.** Every turn it re-reads the whole desk. The desk has a fixed size.
> What is not on the desk does not exist for the model.
>
> **Context management** is deciding what goes on the desk, what comes off, and in what form,
> *before* things drop off on their own.

| On the desk | In the system | Today |
|---|---|---|
| The desk | the context window: everything sent in one request | Part 1 |
| The brief pinned in the corner | system prompt and task | Part 1, 3 |
| Binders piled on top | tool results | Part 1, 2 |
| A sticky note: "binder is on shelf 3" | offloading, clearing old tool results | Part 2 |
| A one-page summary | compaction | Part 3 |
| The drawer | memory: notes kept outside the window | Part 4 |
| An intern sent to the archive | a sub-agent with its own desk | Part 6 (optional) |

**Four moves:** *write* (drawer), *select* (fetch three pages, not the library), *compress* (summary, clearing),
*isolate* (intern). **Cheapest first:** offload at the door → clear → compact → reset with notes.

⏱ **Core: Parts 1–5 (~65 min).** Parts 6–7 are optional, for after class.

## 0. Setup (2 minutes)

You did the setup at home (`SETUP.md`). This cell checks it again; everything should be ✅.

This week runs on **OpenAI**: the techniques work directly on OpenAI's list of conversation items.
(Claude has the same ideas built into its API: *context editing* and *compaction*.)

In [ ]:
%run ../check_setup.py --no-api

In [ ]:
import json, os, re, time, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Literal

from pydantic import BaseModel, Field

import agent_core, lifedb, life_tools
from agent_core import run_agent, tool
from providers import load_api_key, open_conversation

PROVIDER, MODEL = "openai", "gpt-6-luna"
load_api_key(PROVIDER)
from openai import OpenAI
client = OpenAI()

warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")   # harmless SDK noise
DB = lifedb.create_db()          # Alex's week (dated to this week) + a year of history
WEEK = lifedb.week()
SYSTEM = (
    "You are a personal assistant for Alex, a 44-year-old software developer with a full-time job and two kids, "
    f"who is retraining to become an AI engineer. Today is {WEEK.today}. "
    "Use the tools to look at Alex's calendar, tasks and notes; never guess. Reply briefly."
)

print(f"Ready: {MODEL}, Alex's week {WEEK.label(0)} – {WEEK.label(6)}")

## 1. Measure the desk before you manage it

What is actually on the desk? The **system prompt**, the **tool definitions**, the **conversation**, the model's
**tool calls**, and the **tool results** (the binders). Let's count each zone.

(~4 characters ≈ 1 token: rough, but good enough to see proportions. The exact number of every call is in
`usage.input_tokens`, which we also print.)

In [ ]:
def est_tokens(x) -> int:
    """Rough count: about 4 characters per token."""
    if hasattr(x, "model_dump"):
        x = x.model_dump(exclude_none=True)
    if isinstance(x, dict):                       # opaque blobs (encrypted reasoning/compaction): not readable text
        x = {k: v for k, v in x.items() if k not in ("encrypted_content", "id", "status")}
    text = x if isinstance(x, str) else json.dumps(x, ensure_ascii=False, default=str)
    return max(1, len(text) // 4)


def zone_of(item) -> str:
    kind = item.get("type") if isinstance(item, dict) else getattr(item, "type", None)
    if kind == "function_call_output":
        return "tool results"
    if kind in ("function_call", "reasoning"):
        return "tool calls"
    return "conversation"


def desk(conv) -> dict:
    """Tokens per zone of everything the model will read on its next call."""
    zones = {"system prompt": est_tokens(conv.system), "tool definitions": est_tokens(conv.tools),
             "conversation": 0, "tool calls": 0, "tool results": 0}
    for item in conv.input:
        zones[zone_of(item)] += est_tokens(item)
    return zones


def show_desk(conv, api_tokens=None, width=40):
    zones = desk(conv)
    total = sum(zones.values())
    for zone, n in zones.items():
        print(f"{zone:17} {n:7,} tok  {n / total:4.0%} {'█' * round(width * n / total)}")
    print(f"{'total (estimate)':17} {total:7,} tok" + (f"   (the API counted {api_tokens:,} on the last call)" if api_tokens else ""))

**The ladder** (cheapest first). Our desk is **tiny on purpose** (6,000 tokens) so you can watch it fill up.
Real models have 400k+ tokens, but agents fill them the same way, just later, and pay for every token on every step.

In [ ]:
class ContextBudget:
    def __init__(self, window=6_000, reserve_output=1_000):
        self.window, self.reserve_output = window, reserve_output   # keep room for the answer

    def pressure(self, used: int) -> float:
        return used / (self.window - self.reserve_output)

    def action(self, used: int) -> str:
        p = self.pressure(used)
        if p < 0.50: return "OK"
        if p < 0.70: return "CLEAR old tool results"
        if p < 0.85: return "COMPACT older history"
        return "RESET: notes to memory, fresh desk"


BUDGET = ContextBudget()

REVIEW = (f"Review my last 8 weeks, one week at a time (each Monday to Sunday, the first one starting "
          f"{WEEK.day(-56)}). For each week: hours spent in AI course sessions, number of networking chats, "
          "number of gym sessions. Look up exactly one week per tool call, and before the next lookup write "
          "that week's three numbers in one short line (your notes). "
          "Then tell me in two sentences which week was the least balanced.")

Alex asks for an 8-week review. The assistant uses a careless tool that returns **everything** as JSON (a binder per week).
`before_call` runs before every model call: we measure the desk there.

(`parallel_tool_calls: False` makes the model look up one week per step, so we can watch the desk grow step by step.)

In [ ]:
def tracker(log):
    def before_call(conv, turn):
        used = sum(desk(conv).values())
        log.append(used)
        print(f"  step {turn:2}: desk {used:6,} tok · pressure {BUDGET.pressure(used):4.0%} → {BUDGET.action(used)}")
    return before_call

log_plain = []
r_plain = run_agent(REVIEW, life_tools.BINDER_TOOLS, system=SYSTEM, model=MODEL, max_turns=15, verbose=False,
                    extra={"parallel_tool_calls": False}, before_call=tracker(log_plain))
print("\n" + r_plain.answer + "\n")
show_desk(r_plain.conv, api_tokens=r_plain.steps[-1]["input"])

**What's on the desk at the end?** Almost all of it is binders: tool results the model has already read.
(Anthropic measured the same thing on a research agent: 96% of its context was file contents it had already read.)

Is the answer right? Plain SQL knows:

In [ ]:
def ground_truth(weeks=8):
    for k in range(weeks, 0, -1):
        start, end = WEEK.day(-7 * k), WEEK.day(-7 * k + 6)
        rows = life_tools.db_rows("SELECT title FROM events WHERE date BETWEEN ? AND ?", (start, end))
        course = 1.5 * sum(r["title"].startswith("AI course") for r in rows)
        chats = sum(r["title"] == "Networking chat" for r in rows)
        gym = sum(r["title"] == "Gym" for r in rows)
        print(f"week of {start}: course {course:4.1f} h · networking {chats} · gym {gym}")

In [ ]:
ground_truth()

**Discuss:** At which step did the ladder say CLEAR? COMPACT? Did the desk pass 100%? On a real model that's an
error, or worse, the oldest papers silently fall off.

## 2. Sticky notes, not binders (offload and clear)

Two techniques that cost **no model call**:

| Technique | Desk | How |
|---|---|---|
| **Offload at the door** | "binder is on shelf 3" | A big tool output is saved to a file; the model gets a *handle* + a preview, and a `grep_file` tool to search it |
| **Clear old tool results** | file the binders you've read | Old tool results are replaced by `[cleared ...]`. The *call* stays, so the model knows it happened and can call again |

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# everything up to Part 2 (the same code as the cells above), so you can continue from here.
# from checkpoint import *

In [ ]:
OFFLOAD_DIR = Path("desk_files")


def offload_if_large(name: str, output: str, max_tokens: int = 150) -> str:
    if est_tokens(output) <= max_tokens:
        return output
    OFFLOAD_DIR.mkdir(exist_ok=True)
    handle = f"{name}-{len(list(OFFLOAD_DIR.glob('*.txt'))) + 1}.txt"
    (OFFLOAD_DIR / handle).write_text(output, encoding="utf-8")
    preview = "\n".join(output.splitlines()[:6])
    return (f"[{est_tokens(output):,} tokens saved as '{handle}'. Use grep_file(handle, pattern) to search it.]\n"
            f"{preview}\n...")


class GrepIn(BaseModel):
    handle: str = Field(description="The handle of a saved output, e.g. 'list_events-1.txt'")
    pattern: str = Field(description="Regular expression, e.g. 'Gym' or 'AI course'")


def grep_file(handle: str, pattern: str) -> str:
    path = (OFFLOAD_DIR / handle).resolve()
    if path.parent != OFFLOAD_DIR.resolve() or not path.exists():
        raise ValueError(f"No saved output called '{handle}'.")
    hits = [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if re.search(pattern, line, re.I)]
    return f"{len(hits)} matching lines\n" + "\n".join(hits[:30])


def offloaded(tools: dict) -> dict:
    wrapped = {name: {**e, "fn": (lambda _fn=e["fn"], _name=name, **kw: offload_if_large(_name, _fn(**kw)))}
               for name, e in tools.items()}
    wrapped["grep_file"] = tool(grep_file, input_model=GrepIn, description=
        "Search a saved tool output (by its handle) for lines matching a regex pattern. "
        "Returns how many lines match, and the lines. Search several things at once with |, e.g. 'Gym|AI course'.")
    return wrapped


CLEARED = ("[cleared to save space: you already read this result and wrote down what you need in your notes. "
           "Call the tool again only if you need a detail that is not in your notes.]")


def clear_old_tool_results(conv, keep: int = 2, never=("memory",)) -> None:
    """Replace all but the newest `keep` tool results. The calls stay on the desk."""
    names = {i.call_id: i.name for i in conv.input if getattr(i, "type", None) == "function_call"}
    results = [i for i in conv.input if isinstance(i, dict) and i.get("type") == "function_call_output"
               and names.get(i["call_id"]) not in never]
    for item in results[:-keep] if keep else results:
        item["output"] = CLEARED

Same review, three ways. (Each run takes about a minute.)

In [ ]:
def run_review(tools, manage=None, label=""):
    log = []
    def before_call(conv, turn):
        if manage:
            manage(conv)
        log.append(sum(desk(conv).values()))
    r = run_agent(REVIEW, tools, system=SYSTEM, model=MODEL, max_turns=30, verbose=False,
                  extra={"parallel_tool_calls": False}, before_call=before_call)
    return {"label": label, "log": log, "result": r}


def bars(run, scale=150):
    limit = BUDGET.window - BUDGET.reserve_output
    print(f"\n{run['label']}  (peak {max(run['log']):,} tokens, {run['result'].turns} steps)")
    for i, n in enumerate(run["log"], 1):
        print(f"  step {i:2} {n:6,} {'█' * (n // scale)}{'  ⚠ over the desk' if n > limit else ''}")

In [ ]:
runs = [
    run_review(life_tools.BINDER_TOOLS, label="A. no management"),
    run_review(life_tools.BINDER_TOOLS, manage=lambda c: clear_old_tool_results(c, keep=2), label="B. clear old results (keep 2)"),
]
for run in runs:
    bars(run)

In [ ]:
for run in runs:
    r = run["result"]
    print(f"── {run['label']}: {r.turns} steps, {r.tool_calls.count('list_events_raw')} lookups, "
          f"{r.input_tokens:,} input tokens in total\n{r.answer}\n")
ground_truth()

**(Optional) C. Offload at the door.** Every binder goes to a file; the model gets a handle and a preview, and must
`grep_file` for what it needs. Watch the number of steps: offloading trades desk space for **extra calls**.
It pays off when outputs are huge (logs, files, web pages) and the model needs only a few lines of each.

In [ ]:
run_c = run_review(offloaded(life_tools.BINDER_TOOLS), manage=lambda c: clear_old_tool_results(c, keep=2),
                   label="C. offload at the door + clear")
bars(run_c)
print(run_c["result"].tool_calls)
print(run_c["result"].answer)

**Discuss:**
- Which run kept the desk smallest? Which got the numbers right? Cheaper is only better if the answer survives.
- **Lookups:** 8 means the model trusted its notes; more means it fetched cleared weeks again. Run B twice:
  the same code can give either. Clearing is safe *because* the tool can be called again, but each re-fetch costs a step.
- Clearing only works because the model **wrote its notes** (one line per week) before the binders were cleared.
  Remove "write that week's three numbers..." from `REVIEW` and run B again: what happens? (Write before you clear.)
- Clearing is free and **reversible**: the model can call the tool again. That's why it's the first rung of the ladder.

## 3. The one-page summary (sliding window and compaction)

Agents fill the desk with tool output. **Chatbots** fill it with conversation. Alex chats with the assistant all
week. Some things said on Monday matter on Thursday.

We replay Alex's week (10 messages), then ask 5 **probe questions** about things said earlier.
Three strategies: keep everything / sliding window / compaction.

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# everything up to Part 3 (the same code as the cells above), so you can continue from here.
# from checkpoint import *

In [ ]:
CHAT_SYSTEM = (
    "You are Alex's personal assistant. Alex is a 44-year-old developer with a full-time job and two kids, "
    f"retraining as an AI engineer. This chat runs through the week that starts on {WEEK.today}; "
    "every message starts with its day and time. Keep answers short: at most 80 words."
)

ALEX_WEEK = [
    ("Mon 07:45", "Morning! Context for this week: my partner is travelling Wednesday and Thursday, so I'm on my own "
                  "with the kids. No calls between 18:00 and 20:00 on those days."),
    ("Mon 12:30", "Lunch break. In what order should I tackle my capstone tasks? The RAG pipeline scares me."),
    ("Mon 21:10", "Coffee chat with Priya went great. She'll refer me to Kestrel Labs if I send her my resume by Friday."),
    ("Tue 12:10", "Give me 3 good questions to ask the Northwind recruiter tonight."),
    ("Tue 21:15", "Recruiter screen done! Next step: a take-home assignment, due next Wednesday, about 5 hours of work."),
    ("Wed 07:10", "Dentist in an hour. Honestly, what should I prioritise today?"),
    ("Wed 22:00", "Mock interview feedback: my STAR stories are too long. I should aim for 2 minutes each."),
    ("Thu 06:40", "Skipping the gym this morning to work on the RAG pipeline. Bad idea?"),
    ("Thu 21:30", "The RAG pipeline works! Retrieval accuracy went from 62% to 81% after I added reranking."),
    ("Thu 22:15", "Draft 3 bullet points for tomorrow's demo intro, please."),
]

PROBES = [
    ("(Fri 08:00) Remind me: on which evenings was I unavailable for calls this week, and from when to when?",
     lambda a: "18" in a and "20" in a),
    ("(Fri 08:05) By when do I need to send my resume to Priya?", lambda a: "friday" in a or "today" in a),
    ("(Fri 08:10) When is the Northwind take-home due, and how long will it take?",
     lambda a: "wednesday" in a and ("5" in a or "five" in a)),
    ("(Fri 08:15) What was the feedback from my mock interview?", lambda a: "2 minutes" in a or "two minutes" in a),
    ("(Fri 08:20) What retrieval accuracy did my RAG pipeline reach?", lambda a: "81" in a),
]


class ChatSession:
    """A chatbot: the desk is the list of messages. A strategy may tidy it before every call."""

    def __init__(self, strategy=None, extra=None, native=False):
        self.messages, self.strategy, self.extra, self.native, self.log = [], strategy, extra or {}, native, []

    def say(self, text: str) -> str:
        self.messages.append({"role": "user", "content": text})
        if self.strategy:
            self.messages = self.strategy(self.messages)
        r = client.responses.create(model=MODEL, instructions=CHAT_SYSTEM, input=self.messages,
                                    max_output_tokens=2000, reasoning={"effort": "low"}, **self.extra)
        cached = getattr(r.usage.input_tokens_details, "cached_tokens", 0) or 0
        self.log.append({"input": r.usage.input_tokens, "cached": cached})
        if self.native:
            self.messages += r.output          # keep the API's own items (needed for server-side compaction)
            last = max((i for i, m in enumerate(self.messages) if getattr(m, "type", None) == "compaction"), default=0)
            self.messages = self.messages[last:]   # everything before the newest compaction item is folded into it
        else:
            self.messages.append({"role": "assistant", "content": r.output_text})
        return r.output_text


def play_and_probe(session, label):
    for when, text in ALEX_WEEK:
        session.say(f"({when}) {text}")
    score, answers = 0, []
    for question, check in PROBES:
        answer = session.say(question)
        ok = check(answer.lower())
        score += ok
        answers.append(f"  {'✅' if ok else '❌'} {question}\n     → {answer.strip()[:150]}")
    peak = max(x["input"] for x in session.log)
    total = sum(x["input"] for x in session.log)
    print(f"── {label}: probes {score}/5 · biggest desk {peak:,} tok · {total:,} input tokens in total")
    print("\n".join(answers))
    return {"label": label, "score": score, "peak": peak, "total": total}

**Strategy 1: sliding window.** Keep the newest messages that fit, **pin** what must never drop
(here: Alex's first message, the brief), and **say what was dropped**.

In [ ]:
GAP = "[Earlier messages were removed to save space.]"


def sliding_window(messages, budget=500, pin_first=1):
    messages = [m for m in messages if m.get("content") != GAP]          # old markers out
    pinned = messages[:pin_first]
    budget -= sum(est_tokens(m) for m in pinned)
    kept = []
    for m in reversed(messages[pin_first:]):
        if est_tokens(m) > budget:
            break
        kept.insert(0, m)
        budget -= est_tokens(m)
    # (agents) never keep a tool result whose call fell off: the API rejects it
    while kept and isinstance(kept[0], dict) and kept[0].get("type") == "function_call_output":
        kept.pop(0)
    dropped = len(messages) - len(pinned) - len(kept)
    return pinned + ([{"role": "user", "content": GAP}] if dropped else []) + kept

**Strategy 2: compaction.** When the desk gets full, a model writes a **one-page summary** of the older messages.
The newest messages stay word for word. The summary is **incremental**: the previous summary goes in too,
so knowledge accumulates instead of blurring. **The prompt is the product:** name what must survive.

In [ ]:
SUMMARY_PROMPT = """Summarise this conversation so the assistant can continue WITHOUT the original messages.
Keep: 1) Alex's goals and constraints 2) decisions and why 3) exact facts: numbers, dates, names, deadlines
4) what is done and what is still open. Drop chit-chat. At most 150 words."""
SUMMARY_TAG = "[Summary of the conversation so far]"


class Compactor:
    def __init__(self, threshold=600, keep_last=4):
        self.summary, self.threshold, self.keep_last = "", threshold, keep_last

    def __call__(self, messages):
        if sum(est_tokens(m) for m in messages) < self.threshold:
            return messages
        head, tail = messages[:-self.keep_last], messages[-self.keep_last:]
        head = [m for m in head if not m["content"].startswith(SUMMARY_TAG)]   # the old summary goes in below
        transcript = "\n".join(f"{m['role'].upper()}: {m['content']}" for m in head)
        r = client.responses.create(
            model=MODEL, max_output_tokens=2000, reasoning={"effort": "low"},
            input=f"{SUMMARY_PROMPT}\n<previous>{self.summary}</previous>\n<transcript>\n{transcript}\n</transcript>")
        self.summary = r.output_text.strip()
        return [{"role": "user", "content": f"{SUMMARY_TAG}\n{self.summary}"}] + tail

Now the experiment. (About 2 minutes: 3 × 15 messages.)

In [ ]:
compactor = Compactor()
chat_results = [
    play_and_probe(ChatSession(), "1. keep everything"),
    play_and_probe(ChatSession(strategy=sliding_window), "2. sliding window"),
    play_and_probe(ChatSession(strategy=compactor), "3. compaction"),
]

In [ ]:
print(f"{'strategy':22} {'probes':>6} {'biggest desk':>13} {'total input':>12}")
for c in chat_results:
    print(f"{c['label']:22} {c['score']:>4}/5 {c['peak']:>11,} t {c['total']:>10,} t")
print("\nThe final summary:\n" + compactor.summary)

**Discuss:**
- Which facts did the sliding window lose? Which survived, and why? (Hint: the pinned message.)
- Read the summary. Is every probe fact in it? If not, change `SUMMARY_PROMPT` and rerun.
- **Poisoning:** if the summary gets a fact wrong, every later answer builds on it. A wrong fact that survives
  compaction becomes "truth". (Try it in Part 7.)

## 4. The drawer (memory)

Compaction and clearing work **inside** one conversation. When the conversation ends, the desk is cleared.
**Memory** is the only thing that survives: notes in a drawer, outside the window. The model decides what to
write down, through a `memory` tool. We host the storage (files).

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# everything up to Part 4 (the same code as the cells above), so you can continue from here.
# from checkpoint import *

In [ ]:
class FileMemory:
    def __init__(self, root="memories"):
        self.root = Path(root).resolve()
        self.root.mkdir(exist_ok=True)

    def _path(self, p: str) -> Path:                   # block "../../somewhere/else"
        full = (self.root / p.strip().lstrip("/").removeprefix("memories/")).resolve()
        if not full.is_relative_to(self.root):
            raise PermissionError(f"'{p}' is outside the memory folder")
        return full

    def handle(self, command: str, path: str = "", text: str = "", query: str = "") -> str:
        if command == "view":
            target = self._path(path or ".")
            if target.is_dir():                        # the whole drawer: every note, with its content
                notes = [f"## {f.relative_to(self.root)}\n{f.read_text(encoding='utf-8')[:1000]}"
                         for f in sorted(target.rglob("*")) if f.is_file()]
                return "\n\n".join(notes) or "(memory is empty)"
            return target.read_text(encoding="utf-8") if target.exists() else f"No memory file '{path}'."
        if command == "write":
            target = self._path(path)
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_text(text, encoding="utf-8")
            return f"Saved {target.relative_to(self.root)}"
        if command == "search":
            hits = [f"{f.relative_to(self.root)}: {line}" for f in self.root.rglob("*") if f.is_file()
                    for line in f.read_text(encoding="utf-8").splitlines() if query.lower() in line.lower()]
            return "\n".join(hits) or "(no matches)"
        if command == "delete":                        # the right to be forgotten
            self._path(path).unlink(missing_ok=True)
            return f"Deleted {path}"
        raise ValueError("command must be view, write, search or delete")


class MemoryIn(BaseModel):
    command: Literal["view", "write", "search", "delete"]
    path: str = Field("", description="A file in the memory folder, e.g. 'family.md'. Empty with view = list all files.")
    text: str = Field("", description="For write: the complete new content of the file")
    query: str = Field("", description="For search: the text to look for")


MEMORY = FileMemory()


def memory(command, path="", text="", query=""):
    return MEMORY.handle(command, path, text, query)


MEMORY_TOOL = {"memory": tool(memory, input_model=MemoryIn, description=
    "Alex's long-term memory (a folder of notes). ALWAYS view it first: view with no path shows every note. "
    "Write down anything you will need "
    "in a later conversation: the desk is cleared after every conversation.")}

In [ ]:
import shutil
shutil.rmtree("memories", ignore_errors=True); MEMORY = FileMemory()          # start with an empty drawer

try:
    MEMORY.handle("view", "../../check_setup.py")
except PermissionError as exc:
    print("Path guard works:", exc)

**Monday, conversation 1.** Alex mentions something that matters later in the week.

In [ ]:
r1 = run_agent("Heads-up for this week: my partner is travelling Wednesday and Thursday, so I'm doing the "
               "school drop-off at 8:00 on both days. I can't take anything between 7:30 and 8:30 then. "
               "Please remember that.",
               {**life_tools.TOOLS, **MEMORY_TOOL}, system=SYSTEM, model=MODEL)
print("\n" + r1.answer)
print("\n📂 In the drawer:\n" + MEMORY.handle("view"))

**Thursday, conversation 2: a fresh desk.** Same question, with and without the drawer.

In [ ]:
QUESTION = "Northwind asked for a quick call on Thursday from 7:45 to 8:15, before work. Can I say yes?"

r_no_memory = run_agent(QUESTION, life_tools.TOOLS, system=SYSTEM, model=MODEL, verbose=False)
r_memory = run_agent(QUESTION, {**life_tools.TOOLS, **MEMORY_TOOL}, system=SYSTEM, model=MODEL, verbose=False)
print("── without memory:", r_no_memory.tool_calls, "\n" + r_no_memory.answer)
print("\n── with memory:   ", r_memory.tool_calls, "\n" + r_memory.answer)

**Discuss:**
- Did the assistant look in the drawer first? What did it write, and in which file?
- Remove "Please remember that." and run conversation 1 a few times. Does it always write a note? When the model
  decides what to remember, sometimes it doesn't. What would you have the *app* always save instead?
- **Summary vs memory:** a summary is in-session and lossy; memory is cross-session and exact, for what was saved.
- **Privacy:** memory is personal data. It needs a purpose, a retention period and a way to forget
  (`command="delete"`). Memory is also an attack surface: notice the path guard.

## 5. Don't hand-roll everything: what the OpenAI API does for you

| Feature | In the API | You still |
|---|---|---|
| **Prompt caching** | automatic: a repeated *prefix* is billed at 0.1×; see `usage.input_tokens_details.cached_tokens` | keep instructions and tools identical and **first** |
| **Server-side compaction** | `context_management=[{"type": "compaction", "compact_threshold": N}]` | append the output items as they come |
| **On-demand compaction** | `client.responses.compact(model=..., input=...)` | pass the result back as-is |
| **Clearing, memory** | not built in | bring Parts 2 and 4 as function tools |

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# everything up to Part 5 (the same code as the cells above), so you can continue from here.
# from checkpoint import *

### 5a. Prompt caching: keep the beginning of the desk stable

How much of each call in Part 2 came from the cache? Then the same review with a **timestamp at the top of the
system prompt**, a classic mistake: the start of the desk changes on every call, so nothing can be cached.

In [ ]:
PRICE = {"input": 0.10, "cached": 0.01, "output": 0.50}      # gpt-6-luna, $ per 1M tokens


def cost(r) -> float:
    uncached = r.input_tokens - r.cached_tokens
    return (uncached * PRICE["input"] + r.cached_tokens * PRICE["cached"] + r.output_tokens * PRICE["output"]) / 1e6


def stamp(conv, turn):
    conv.system = f"Current time: {time.time():.3f}\n" + SYSTEM     # changes every call: breaks the cache

In [ ]:
r_stable = run_agent(REVIEW, life_tools.BINDER_TOOLS, system=SYSTEM, model=MODEL, max_turns=20, verbose=False,
                     extra={"parallel_tool_calls": False})
r_stamped = run_agent(REVIEW, life_tools.BINDER_TOOLS, system=SYSTEM, model=MODEL, max_turns=20, verbose=False,
                      extra={"parallel_tool_calls": False}, before_call=stamp)
for label, r in [("stable prefix", r_stable), ("timestamp first", r_stamped)]:
    print(f"{label:16} input {r.input_tokens:6,} · cached {r.cached_tokens:6,} "
          f"({r.cached_tokens / max(r.input_tokens, 1):4.0%}) · ${cost(r):.5f}")
print("\nper step (stable):", [(s['input'], s['cached']) for s in r_stable.steps])

**Discuss:** Clearing (Part 2) and compaction (Part 3) also change the desk *before* the newest message.
What does that do to the cache? Look at run B: `runs[1]["result"].steps`. This is the trade-off between a small desk
and a cached desk.

### 5b. Compaction by the API

The same chat as Part 3, with **server-side compaction**: the server folds older history into one opaque
`compaction` item when the desk crosses the threshold. We keep the API's own items (`native=True`).

In [ ]:
server = ChatSession(native=True, extra={"store": False,
                                         "context_management": [{"type": "compaction", "compact_threshold": 1000}]})
try:
    server_result = play_and_probe(server, "4. server-side compaction")
    kinds = [getattr(m, "type", None) or m.get("type") or m.get("role") for m in server.messages]
    print("\nitem types on the desk:", kinds)
except Exception as exc:
    print("Server-side compaction is not available for this model/account:", type(exc).__name__, str(exc)[:200])

In [ ]:
# On-demand: compact a long window yourself, at a moment you choose (e.g. between two tasks).
long_chat = ChatSession()
for when, text in ALEX_WEEK:
    long_chat.say(f"({when}) {text}")
try:
    compacted = client.responses.compact(model=MODEL, input=long_chat.messages)
    question = [{"role": "user", "content": "(Fri 08:00) What did Priya promise me?"}]
    for label, window in [("full window", long_chat.messages), ("compacted window", list(compacted.output))]:
        r = client.responses.create(model=MODEL, instructions=CHAT_SYSTEM, input=window + question,
                                    max_output_tokens=2000, reasoning={"effort": "low"})
        print(f"{label:17} {len(window):2} items · the API counted {r.usage.input_tokens:,} input tokens → {r.output_text[:90]}")
    print("item types after compaction:", [getattr(m, "type", "?") for m in compacted.output])
except Exception as exc:
    print("responses.compact is not available for this model/account:", type(exc).__name__, str(exc)[:200])

**Discuss:** The compaction item is **encrypted**: you can't read or correct the summary. When must a detail
survive for sure? (Answer: write it to memory, Part 4.) When would you hand-roll compaction anyway?

## 6. (Optional) Send an intern to the archive (sub-agents)

**Isolate:** each week of the review goes to an "intern" with its own empty desk. The intern reads the binder and
returns **one small page** (a contract: structured JSON). The main agent only ever sees 8 small pages.

In [ ]:
# 🧭 Checkpoint. Did something above fail? Uncomment the next line and run it: it loads
# everything up to Part 6 (the same code as the cells above), so you can continue from here.
# from checkpoint import *

In [ ]:
WEEK_PAGE = {
    "type": "object",
    "properties": {"course_hours": {"type": "number"}, "networking_chats": {"type": "integer"},
                   "gym_sessions": {"type": "integer"}, "note": {"type": "string"}},
    "required": ["course_hours", "networking_chats", "gym_sessions", "note"],
    "additionalProperties": False,
}


def run_subagent(k: int) -> dict:
    start, end = WEEK.day(-7 * k), WEEK.day(-7 * k + 6)
    binder = life_tools.list_events_raw(start, end)                                # the messy work
    intern = open_conversation(PROVIDER, MODEL, f"Events of the week {start} to {end}:\n{binder}", effort="low",
                               system="Count: hours of 'AI course' sessions (1.5 h each), number of 'Networking chat' "
                                      "events, number of 'Gym' events. note: one short sentence about the week.")
    page = {"week": start, **intern.complete_json(WEEK_PAGE)}
    assert est_tokens(page) <= 150, "the contract: one small page"
    return page

In [ ]:
with ThreadPoolExecutor(max_workers=4) as pool:
    pages = list(pool.map(run_subagent, range(8, 0, -1)))
for p in pages:
    print(p)
final = client.responses.create(model=MODEL, reasoning={"effort": "low"},
                                input="Which of these weeks was the least balanced between the AI course, networking "
                                      f"and the gym? Two sentences.\n{json.dumps(pages, indent=1)}")
print(f"\nThe orchestrator's desk: ≈ {est_tokens(pages):,} tokens of pages "
      f"(vs ≈ {max(runs[0]['log']) if 'runs' in globals() else '?'} tokens for one agent reading every binder)\n")
print(final.output_text)

**Discuss:** Total spend goes *up* (each intern pays for its own desk), but the main desk stays small and the interns
run in parallel. When is that worth it? When is it not (tightly coupled work)?

## 7. (Optional) Poisoning: a wrong fact on the desk

Corrupt the summary from Part 3 on purpose, then ask a probe question. Does the assistant trust the summary over
common sense? How would you defend against this? (Ideas: keep source facts in memory; re-check facts with tools;
test your summary prompt with probe questions, like we did.)

In [ ]:
poisoned_summary = compactor.summary.replace("81%", "91%")              # one wrong number in the summary
print("poisoned:", poisoned_summary != compactor.summary)
poisoned = ChatSession()
poisoned.messages = [{"role": "user", "content": f"{SUMMARY_TAG}\n{poisoned_summary}"},
                     {"role": "assistant", "content": "Got it."}]
print(poisoned.say("(Fri 09:00) I'm writing my demo slides. What retrieval accuracy should I put on the results slide?"))

## 8. Takeaways and homework

**The one-minute version** (explain it to anyone): *It only sees what's on the desk. Every turn it re-reads it all:
more paper, more cost. Pin the brief in the corner. Leave a sticky note, not the binder. Before papers fall off,
summarise what matters. Keep tomorrow's notes in a drawer. Bring three pages, not the library. Send an intern
to the archive.*

**Six habits:** measure first · pin and mark · cheapest first (offload → clear → compact → reset) ·
write it down · isolate the mess · govern it (security, privacy, evals).

### Homework (required)

Turn the notebook into a package, with tests that run **without an API key**:

```
context_toolkit/
├── budget.py        # est_tokens, desk zones, ContextBudget (the ladder)
├── window.py        # sliding_window: pin, safe boundaries, gap marker
├── compaction.py    # Compactor: incremental, keeps the tail; the summariser is a parameter (fake it in tests)
├── tool_results.py  # offload_if_large + grep_file, clear_old_tool_results (never "memory")
├── memory.py        # FileMemory + the memory tool; path guard
└── subagents.py     # run_subagent with a size contract
tests/               # one test file per module
```

Must-have tests: the ladder's four thresholds; the window keeps the pinned message, never starts with a tool
result, adds exactly one gap marker; compaction leaves the last 4 messages untouched and passes the previous
summary in; clearing keeps the newest N results and never touches `memory`; the memory path guard blocks `../`.

### Stretch
- **The full ladder:** one `ContextManager` used as `before_call` that measures, then clears, then compacts,
  then resets (notes to memory, fresh desk), like the 40-contracts demo.
- **Pinned facts:** a list that compaction may never drop; compare probe scores.
- **Measure the cache cost of clearing:** run B vs A, cached tokens and dollars.